# Chapter 3: Handling Sequences with PyTorch
**Module 02: Intermediate Deep Learning with PyTorch**  
*Instructor: Michal Oleszak, Machine Learning Engineer*

> Complete PDF-integrated notebook for sequential data, time-series windows, RNN/LSTM/GRU models, tensor reshaping, and forecasting evaluation.


## Learning Objectives
- Describe sequential data and why order matters.
- Split time series by time to avoid look-ahead bias.
- Create fixed-length input sequences and next-step targets.
- Convert sequences to `TensorDataset` and `DataLoader` objects.
- Build RNN, LSTM, and GRU regressors.
- Train and evaluate recurrent models with mean squared error.


## 1. Sequential Data
Sequential data is ordered in time or space, and that order contains dependencies.

Examples from the PDF:
- Time series
- Text
- Audio waves

The chapter task is electricity-consumption prediction: predict future consumption from past patterns.


## 2. Train-Test Split
> **Never randomly split time series data.**

Random splitting creates **look-ahead bias**, where a model gets information from the future. Split by time instead: train on earlier observations, test on later observations.


In [ ]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim

DATA_DIR = Path("datasets")
ELECTRICITY_ZIP = DATA_DIR / "electricity_consump.zip"
ELECTRICITY_DIR = DATA_DIR / "electricity_consump"
if ELECTRICITY_ZIP.exists() and not ELECTRICITY_DIR.exists():
    with zipfile.ZipFile(ELECTRICITY_ZIP) as zf:
        zf.extractall(DATA_DIR)

train_csv = ELECTRICITY_DIR / "electricity_train.csv"
test_csv = ELECTRICITY_DIR / "electricity_test.csv"
if train_csv.exists() and test_csv.exists():
    train_data = pd.read_csv(train_csv)
    test_data = pd.read_csv(test_csv)
else:
    np.random.seed(42)
    timestamps = pd.date_range("2011-01-01", periods=1000, freq="15min")
    consumption = np.sin(np.linspace(0, 50, 1000)) + np.random.normal(0, 0.1, 1000)
    df = pd.DataFrame({"timestamp": timestamps, "consumption": consumption})
    split_idx = int(len(df) * 0.8)
    train_data = df.iloc[:split_idx].reset_index(drop=True)
    test_data = df.iloc[split_idx:].reset_index(drop=True)

print(train_data.head())
print("Train:", train_data.shape, "Test:", test_data.shape)


## 3. Creating Sequences
A sequence length is the number of data points in one training example.

The PDF uses `24 x 4 = 96`, meaning 24 hours of 15-minute readings. The target is the next single point.


In [ ]:
def create_sequences(df, seq_length):
    xs, ys = [], []
    for i in range(len(df) - seq_length):
        x = df.iloc[i:(i + seq_length), 1]
        y = df.iloc[i + seq_length, 1]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

seq_length = 96
X_train, y_train = create_sequences(train_data, seq_length)
X_test, y_test = create_sequences(test_data, seq_length)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)


## 4. `TensorDataset`
`TensorDataset` packages aligned tensors as supervised examples.


In [ ]:
dataset_train = TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train).float())
dataset_test = TensorDataset(torch.from_numpy(X_test).float(), torch.from_numpy(y_test).float())

batch_size = 32
dataloader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(dataset_test, batch_size=batch_size, shuffle=False, drop_last=True)

seqs, labels = next(iter(dataloader_train))
print(seqs.shape, labels.shape)


## 5. Recurrent Architectures
| Architecture | Description | Example |
|---|---|---|
| Sequence-to-sequence | Use the entire output sequence | Real-time speech recognition |
| Sequence-to-vector | Use the last output | Topic classification, forecasting |
| Vector-to-sequence | Single input, sequence output | Text generation |
| Encoder-decoder | Read full input before generating output | Machine translation |


## 6. RNN in PyTorch
A recurrent neuron receives current input `x`, produces output `y`, and updates hidden state `h`. In PyTorch, the basic recurrent layer is `nn.RNN()`.


In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(input_size=1, hidden_size=32, num_layers=2, batch_first=True)
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        h0 = torch.zeros(2, x.size(0), 32, device=x.device)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out

rnn_net = Net()
dummy = torch.randn(batch_size, seq_length, 1)
print(rnn_net(dummy).shape)


## 7. LSTM and GRU Cells
Basic RNN memory is short-term. The PDF introduces two stronger alternatives.

| Cell | State | Key Idea |
|---|---|---|
| LSTM | Hidden state `h` and cell state `c` | Forget, input, and output gates control memory |
| GRU | One hidden state | Simpler than LSTM; less computation |

RNNs are less common for long dependencies; try LSTM and GRU and compare.


In [ ]:
class LSTMNet(nn.Module):
    def __init__(self, input_size=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=32, num_layers=2, batch_first=True)
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        h0 = torch.zeros(2, x.size(0), 32, device=x.device)
        c0 = torch.zeros(2, x.size(0), 32, device=x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

lstm_net = LSTMNet()
print(lstm_net(dummy).shape)


In [ ]:
class GRUNet(nn.Module):
    def __init__(self, input_size=1):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=32, num_layers=2, batch_first=True)
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        h0 = torch.zeros(2, x.size(0), 32, device=x.device)
        out, _ = self.gru(x, h0)
        out = self.fc(out[:, -1, :])
        return out

gru_net = GRUNet()
print(gru_net(dummy).shape)


## 8. Tensor Shapes for Recurrent Layers
Recurrent layers expect:

```text
(batch_size, sequence_length, num_features)
```

The raw batch is `(batch_size, sequence_length)`, so add one final feature dimension.


In [ ]:
for seqs, labels in dataloader_train:
    print("Before:", seqs.shape)
    seqs = seqs.view(batch_size, seq_length, 1)
    print("After: ", seqs.shape)
    break


### Squeezing Outputs
Labels are shaped `(batch_size)`. Model outputs are often `(batch_size, 1)`. Use `squeeze()` so shapes match for loss/metrics.


In [ ]:
for seqs, labels in test_loader:
    seqs = seqs.view(batch_size, seq_length, 1)
    out = lstm_net(seqs)
    print("labels:", labels.shape)
    print("output:", out.shape)
    print("squeezed:", out.squeeze().shape)
    break


## 9. Training and Evaluation
For forecasting a numeric value, use mean squared error.

| Item | PyTorch |
|---|---|
| Loss | `nn.MSELoss()` |
| Optimizer | `optim.Adam` |
| Metric | `torchmetrics.MeanSquaredError()` |


In [ ]:
net = LSTMNet()
criterion = nn.MSELoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

net.train()
for epoch in range(1):
    total_loss = 0.0
    for seqs, labels in dataloader_train:
        seqs = seqs.view(batch_size, seq_length, 1)
        outputs = net(seqs).squeeze()
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} | loss={total_loss/len(dataloader_train):.4f}")


In [ ]:
try:
    import torchmetrics
    mse = torchmetrics.MeanSquaredError()
    net.eval()
    with torch.no_grad():
        for seqs, labels in test_loader:
            seqs = seqs.view(batch_size, seq_length, 1)
            outputs = net(seqs).squeeze()
            mse(outputs, labels)
    print(f"Test MSE: {mse.compute()}")
except ImportError:
    print("torchmetrics is not installed. The PDF metric cell is ready once it is available.")


## Chapter Summary
- Sequence order carries information, so time series require time-based splits.
- Sliding windows convert a time series into supervised learning examples.
- RNN, LSTM, and GRU layers consume `(batch, sequence, features)` tensors.
- LSTM uses hidden and cell states; GRU uses a simpler hidden-state design.
- Forecasting loops use MSE loss and often squeeze model outputs during evaluation.
